# Bank Customer Churn Prediction

Reconstructed reproducible notebook based on the supplied dataset and documented workflow. This is not the original notebook.
Target: `Exited`; stratified 80/20 split; random_state=42.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

df = pd.read_csv('../Data/European_Bank.csv')
df.shape, df.isna().sum().sum(), df.duplicated().sum()

In [ ]:
y = df['Exited'].astype(int)
X = df.drop(columns=['Exited','CustomerId','Surname','Year']).copy()
X['BalanceZero'] = (X.Balance == 0).astype(int)
X['ManyProducts'] = (X.NumOfProducts >= 3).astype(int)
cat = X.select_dtypes(include=['object','category']).columns.tolist()
num = [c for c in X.columns if c not in cat]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=42,stratify=y)
def prep():
    return ColumnTransformer([('num',SimpleImputer(strategy='median'),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
models = {'Logistic Regression':LogisticRegression(max_iter=1000),'Decision Tree':DecisionTreeClassifier(random_state=42),'Random Forest':RandomForestClassifier(n_estimators=300,random_state=42,n_jobs=-1),'Gradient Boosting':GradientBoostingClassifier(random_state=42)}
rows=[]; gb_prob=None
for name, model in models.items():
    pipe=Pipeline([('preprocess',prep()),('model',model)]).fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Churn_Precision':precision_score(y_test,pred,zero_division=0),'Churn_Recall':recall_score(y_test,pred,zero_division=0),'Churn_F1':f1_score(y_test,pred,zero_division=0),'ROC_AUC':roc_auc_score(y_test,prob)})
    if name=='Gradient Boosting': gb_prob=prob
metrics=pd.DataFrame(rows); metrics

In [ ]:
from pathlib import Path
scored=df.loc[X_test.index].copy()
scored['ChurnRisk']=gb_prob
scored['RiskCategory']=pd.cut(gb_prob,[-.000001,.30,.60,1],labels=['Low','Medium','High'],include_lowest=True).astype(str)
scored['ActualChurn']=y_test.values
Path('../outputs').mkdir(exist_ok=True)
metrics.to_csv('../outputs/model_comparison.csv',index=False)
scored.to_csv('../outputs/customer_churn_risk_scores.csv',index=False)
scored.RiskCategory.value_counts()